# Earthquakes

Every earthquake of magnitude 2.5 or more in the last 30 days, from the [US Geological Survey](https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php) (the list is updated every minute). With pandas we load it, sort it, count it and put it on a map.

Tap **Run All** at the top, or run the cells one by one with ▶.

In [ ]:
import io

import pandas as pd
import requests

url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/2.5_month.csv"
csv = requests.get(url, timeout=30).text
quakes = pd.read_csv(io.StringIO(csv), parse_dates=["time"])
quakes = quakes[quakes["type"] == "earthquake"]
print(f"{len(quakes):,} earthquakes in the last 30 days")
quakes[["time", "mag", "depth", "place"]].head()

## The strongest

`nlargest` sorts by a column and keeps the top rows. Depth is in kilometers.

In [ ]:
quakes.nlargest(10, "mag")[["time", "mag", "depth", "place"]]

## Where

Longitude and latitude are enough for a map, even without a map behind it: the dots trace the edges of the plates that make up the Earth's crust, like the Ring of Fire around the Pacific. Bigger dots are stronger; darker ones are deeper.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

fig, ax = plt.subplots(figsize=(10, 5.2))
dots = ax.scatter(
    quakes.longitude,
    quakes.latitude,
    s=3 ** (quakes.mag - 2),
    c=quakes.depth.clip(lower=1),
    norm=LogNorm(),
    cmap="magma_r",
    alpha=0.7,
    linewidths=0,
)
fig.colorbar(dots, label="depth (km)", shrink=0.8)
ax.set(xlim=(-180, 180), ylim=(-70, 80), xlabel="longitude", ylabel="latitude", aspect="equal")
ax.set_facecolor("#eef3f8")
ax.set_title("Earthquakes, last 30 days");

## When

`resample("D")` groups the rows by day. Days with a strong quake often have many more, from its aftershocks.

In [ ]:
per_day = quakes.set_index("time").resample("D").size()

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(per_day.index, per_day.values, width=0.8, color="#0a84ff")
ax.set_ylabel("earthquakes")
fig.autofmt_xdate();

## Small ones are common

The Gutenberg–Richter law says that for each step of magnitude there are about ten times fewer earthquakes: on a log scale, the count of quakes *at least* that strong is a straight line, log₁₀ N = a − b · M, with *b* close to 1.

Far from seismometers the list misses the small ones, so the fit starts at magnitude 4.5.

In [ ]:
import numpy as np

magnitudes = np.arange(2.5, quakes.mag.max() + 0.1, 0.1)
at_least = np.array([(quakes.mag >= m).sum() for m in magnitudes])
fit = (magnitudes >= 4.5) & (at_least >= 5)
slope, a = np.polyfit(magnitudes[fit], np.log10(at_least[fit]), 1)
b = -slope

plt.figure(figsize=(7, 4))
plt.semilogy(magnitudes, at_least, "o", markersize=3, label="earthquakes")
plt.semilogy(magnitudes[fit], 10 ** (a - b * magnitudes[fit]), label=f"fit: b = {b:.2f}")
plt.xlabel("magnitude")
plt.ylabel("at least this strong")
plt.legend()
print(f"b = {b:.2f}: one step of magnitude, {10 ** b:.1f}x fewer earthquakes")

## Your turn

- Other lists: `4.5_week.csv`, `significant_month.csv` or `all_day.csv` in the URL ([all of them](https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php)).
- Only one region: `quakes[quakes.place.str.contains("Japan", na=False)]`.
- How deep are they? `quakes.depth.plot.hist(bins=50)`.